# Lab 08: RNNs, LSTMs, and GRUs for Forecasting

            **Duration:** 3 hours  
            **Lecture alignment:** Week 8 — Recurrent neural networks  
            **CLO mapping:** CLO-1, CLO-2, CLO-3  
            **Framework:** PyTorch (standalone, credential-free, CPU smoke-test with optional GPU)

            ## Learning objectives

            - Create chronological sequence windows without leakage.
- Implement and compare vanilla RNN, LSTM, and GRU models.
- Relate state structure and gradient behavior to forecasting performance.

            ## Three-hour activity plan

            - 0–30 min: time-series split and windowing
- 30–65 min: hidden/cell state shape exercises
- 65–125 min: train three recurrent models
- 125–160 min: forecasts, gradients, cost comparison
- 160–180 min: checks and architecture recommendation


## Book grounding

            - Goodfellow, Bengio, and Courville, *Deep Learning*, MIT Press, 2016.
- Zhang, Lipton, Li, and Smola, *Dive into Deep Learning*, Cambridge University Press, 2024.
- Prince, *Understanding Deep Learning*, MIT Press, 2023.

            The notebook paraphrases concepts and supplies original code; it does not reproduce book text.


In [ ]:
from pathlib import Path
import json, math, os, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

FAST_MODE = True
RUN_EXTENSION = False
SEED = 20268
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(2, os.cpu_count() or 1))

if Path("/content").exists():
    ARTIFACT_DIR = Path("/content/artifacts/lab_08")
else:
    ARTIFACT_DIR = Path.cwd() / "tmp" / "course_build" / "runtime" / "lab_08"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print({"lab": 8, "device": str(DEVICE), "fast_mode": FAST_MODE,
       "artifacts": str(ARTIFACT_DIR), "torch": torch.__version__})


## Predict before running

Which gated recurrent model will achieve the lowest RMSE, and does its extra parameter count justify the difference?

Record a brief prediction in your own words before executing the experiment, then revisit it in the exit reflection.


## Activity 1 — Chronological windows without leakage


In [ ]:
total=900 if FAST_MODE else 3600
t=torch.linspace(0,40,total); series=torch.sin(t)+.35*torch.sin(3.1*t)+.01*t+.05*torch.randn(total)
window=20
X=torch.stack([series[i:i+window] for i in range(total-window)]).unsqueeze(-1)
y=torch.stack([series[i+window] for i in range(total-window)]).unsqueeze(-1)
split=int(.8*len(X)); Xtr,Xte,ytr,yte=X[:split],X[split:],y[:split],y[split:]
assert Xtr[-1,-1,0].item()==series[split-1+window-1].item()
print({"train_windows":tuple(Xtr.shape),"test_windows":tuple(Xte.shape)})


## Activity 2 — Capacity-matched RNN, LSTM, and GRU


In [ ]:
class SequenceRegressor(nn.Module):
    def __init__(self,kind,hidden=14):
        super().__init__(); cell={"RNN":nn.RNN,"LSTM":nn.LSTM,"GRU":nn.GRU}[kind]
        self.recurrent=cell(1,hidden,batch_first=True); self.head=nn.Linear(hidden,1)
    def forward(self,x):
        out,_=self.recurrent(x); return self.head(out[:,-1])
def fit(kind):
    torch.manual_seed(SEED);model=SequenceRegressor(kind).to(DEVICE);opt=torch.optim.Adam(model.parameters(),lr=.015)
    loader=DataLoader(TensorDataset(Xtr,ytr),batch_size=64,shuffle=False);hist=[];grad_hist=[]
    for _ in range(12 if FAST_MODE else 45):
        total=0;model.train()
        for xb,yb in loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE);opt.zero_grad();loss=F.mse_loss(model(xb),yb);loss.backward()
            grad_hist.append(model.recurrent.weight_ih_l0.grad.norm().item());torch.nn.utils.clip_grad_norm_(model.parameters(),2);opt.step();total+=loss.item()*len(xb)
        hist.append(total/len(Xtr))
    model.eval();start=time.perf_counter()
    with torch.no_grad():pred=model(Xte.to(DEVICE)).cpu()
    runtime=time.perf_counter()-start;rmse=torch.sqrt(F.mse_loss(pred,yte)).item()
    return model,hist,rmse,runtime,pred,grad_hist
results={kind:fit(kind) for kind in ("RNN","LSTM","GRU")}
summary={k:{"rmse":v[2],"runtime":v[3],"parameters":sum(p.numel() for p in v[0].parameters()),"median_grad":float(np.median(v[5]))} for k,v in results.items()}
print(json.dumps(summary,indent=2))


## Activity 3 — Forecast comparison


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,3.7))
for kind,value in results.items():axes[0].plot(value[1],label=kind)
axes[0].set(title="Training loss",xlabel="epoch",ylabel="MSE");axes[0].legend()
axes[1].plot(yte[:120],label="target",color="black",linewidth=2)
for kind,value in results.items():axes[1].plot(value[4][:120],label=kind,alpha=.8)
axes[1].set(title="One-step forecasts",xlabel="test window");axes[1].legend(ncol=2)
fig.tight_layout();fig.savefig(ARTIFACT_DIR/"sequence_models.png",dpi=150);plt.show()
(ARTIFACT_DIR/"metrics.json").write_text(json.dumps(summary,indent=2))


## Automated checks


In [ ]:
assert all(v[4].shape==yte.shape for v in results.values())
assert all(math.isfinite(v[2]) and v[1][-1]<v[1][0] for v in results.values())
assert min(v[2] for v in results.values())<.45
assert (ARTIFACT_DIR/"sequence_models.png").exists()
print("All Lab 08 checks passed.")


## Deliverables

                - Leakage-safe windowing assertions
- RNN/LSTM/GRU metric table
- Forecast/learning-curve artifact and recommendation

                Submit the executed notebook and the files created in `/content/artifacts/lab_08/`.


## Disabled extension

The following challenge is intentionally disabled by default so the CPU baseline stays quick.


In [ ]:
if RUN_EXTENSION:
    print("Extension: implement recursive multi-step forecasting and chart error accumulation by horizon.")
else:
    print("Extension disabled: recursive multi-step forecasting or character-level generation.")


## Exit reflection

In 4–6 sentences, state: (1) whether your prediction was supported, (2) the strongest evidence,
(3) one failure mode or limitation, and (4) the next experiment you would run. Include at least
one measured value rather than only a general claim.
